In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Universal Sparse Tensor Exercises

Sparse tensors are vectors, matrices, and higher-dimensional generalizations with many zeros. They are crucial in various fields such as scientific computing, signal processing, and deep learning due to their efficiency in storage, computation, and power. Despite their benefits, handling sparse tensors manually or through existing libraries is often cumbersome, error-prone, nonportable, and does not scale with the combinatorial explosion of sparsity patterns, data types, operations, and targets.

The **Universal Sparse Tensor** (UST) decouples a tensor’s sparsity from its memory storage representation. The UST uses a Domain-Specific Language (DSL) to describe how a tensor should be represented in memory. Developers focus on the sparsity of a tensor only. Compile-time or runtime inspection of the chosen format for the operands in sparsity-agnostic polymorphic operations eventually decides between dispatching to an optimized handwritten library or otherwise with automatic sparse code generation when no such solution exists.

Background material can be found in the blog postings:

 - [Establishing a scalable sparse ecosystem with the UST](https://developer.nvidia.com/blog/establishing-a-scalable-sparse-ecosystem-with-the-universal-sparse-tensor/)
 - [Simplify Sparse Deep Learning with UST in nvmath-python](https://developer.nvidia.com/blog/simplify-sparse-deep-learning-with-universal-sparse-tensor-in-nvmath-python/)

This notebook introduces the UST implementation in nvmath-python.

## Key Topics:

- Introduction to the UST in nvmath-python
- Zero-cost interoperability with sparse packages: data-movement-free conversion with e.g. CuPy
- Common and custom formats: define novel sparsity schemes
- Transparent caches: avoid JIT/LTO recompilation and replanning

## Key Insights:

- Decouples a tensor’s sparsity from its memory storage representation
- DSL describes how a tensor should be represented in memory
- Developers focus on the sparsity of a tensor only

## Sparse Storage Formats in CuPy: DIA, CSR, COO

Before we look at the UST, let's first explore common sparse storage formats in CuPy.

In [ ]:
import cupy as cp
import cupyx.scipy.sparse as sp

# Create a sparse matrix: tridiagonal matrix in DIA, CSR, and COO format.
n = 8
values = cp.array([[-1.0] * n, [4.0] * n, [1.0] * n], dtype=cp.float32)
offsets = cp.array([-1, 0, 1], dtype=cp.int32)
a_cp_dia = sp.dia_matrix((values, offsets), shape=(n, n))
a_cp_csr = sp.csr_matrix(a_cp_dia)
a_cp_coo = sp.coo_matrix(a_cp_dia)
    
a_cp_coo.sum_duplicates()   # coalesce COO format

You can explore the contents of each sparse storage format with a simple print statement. Let's look at the COO format below, which stores every nonzero as coordinate tuple and associated value.

In [ ]:
print(a_cp_coo)

The DIA format and CSR format use different compaction schemes. No single sparse format is optimal for all cases; the best choice depends on the nonzero distribution, operations, and target architecture.

Having different sparse storage formats distracts from the actual coding effort. The UST addresses this by decoupling a tensor’s sparsity from its memory storage representation by using a Domain-Specific Language (DSL) to describe how a tensor should be represented in memory. Developers focus on the sparsity of a tensor only.

## UST in nvmath-python

We have implemented this idea in an experimental part of nvmath-python. A fundamental design decision was to provide zero-cost interoperability between the UST and sparse storage formats in PyTorch, SciPy, and CuPy.

We can construct a UST view of the three sparse storage formats above as follows.

In [ ]:
from nvmath.sparse.ust import Tensor

u_dia = Tensor.from_package(a_cp_dia)
u_csr = Tensor.from_package(a_cp_csr)
u_coo = Tensor.from_package(a_cp_coo)

Once we have a UST view, we can use all the built-in utilities such as printing and drawing.

For example, when we output the COO format, we see the familiar coordinate tuples (in crd) and values (in values). The format is a DSL description of the COO format. You can change the variable below to explore the other two formats and see how the DSL changes. Also note how many data each format uses.

In [ ]:
u_coo

We can also draw the nonzero structure directly as follows.

In [ ]:
u_coo.draw()

And we can also show the storage contents directly. Try this for the other formats as well to see how storage differs.

In [ ]:
u_coo.draw_storage()

## Stateful API for generic matmul with UST

The UST also supports the Stateful APIs for generic matrix multiplication introduced earlier today, where we perform the specification and planning once and amortize their cost through multiple executions. For the UST, the planning also involves determining whether a library implementation is available (the fast-path, i.e. SpMV on CSR is available using cuSPARSE), or whether on-the-fly code generation must be used for cases that lack direct library support (the slow-path, requires JIT/LTO of the new code). Note that the results of JIT/LTO-ing a new kernel (which can have a large cost) are cached between planning phases, so that you only pay the compilation time for a particular format once (making subsequent planning phases take the fast-path again).


Let's first run a SpMV using CuPy and using UST nvmath-python.

In [ ]:
cp_ones = cp.ones((n,), dtype=cp.float32)

In [ ]:
# Use CuPy SpMV.
a_cp_coo @ cp_ones

In [ ]:
from nvmath.sparse.generic import Matmul

# Use UST SpMV.
cp_out = cp.zeros((n,), dtype=cp.float32)
u_ones = Tensor.from_package(cp_ones)
u_out = Tensor.from_package(cp_out)

# Stateful API for generic matmul. Involves planning and execution phase, where planning costs can be amortized over many executions.
# In addition, a JIT/LTO for a novel format is cached even between different planning phases (assuming the format is the same).
with Matmul(u_csr, u_ones, u_out) as mm:
    mm.plan()
    mm.execute()

The proper way to change the UST view back to CuPy land is as follows.

In [ ]:
u_out.to_package()

But since `u_out` is just a view into the buffers of `cp_out`, we can also explore the contents of the CuPy storage directly.

In [ ]:
cp_out

## Benchmark CuPy DIA/CSR/COO with UST DIA/CSR/COO

Now, let's benchmark this. First we make the matrices larger to get meaningful measurements.

In [ ]:
n = 1024 * 512
values = cp.array([[0.0] + [-1.0] * (n - 1), [4.0] * n, [1.0] * (n - 1) + [0.0]], dtype=cp.float32)
offsets = cp.array([-1, 0, 1], dtype=cp.int32)
a_cp_dia = sp.dia_matrix((values, offsets), shape=(n, n))
a_cp_csr = sp.csr_matrix(a_cp_dia)
a_cp_coo = sp.coo_matrix(a_cp_dia)
a_cp_coo.sum_duplicates()   # coalesce COO format

cp_ones = cp.ones((n,), dtype=cp.float32)
cp_out = cp.zeros((n,), dtype=cp.float32)

u_dia = Tensor.from_package(a_cp_dia)
u_csr = Tensor.from_package(a_cp_csr)
u_coo = Tensor.from_package(a_cp_coo)
u_ones = Tensor.from_package(cp_ones)
u_out = Tensor.from_package(cp_out)

Then we benchmark CuPy.

In [ ]:
from cupyx.profiler import benchmark

def matmul(a, x):
    return a @ x

cupy1 = benchmark(matmul, (a_cp_dia, cp_ones), name="CuPy DIA", n_repeat=10)
cupy2 = benchmark(matmul, (a_cp_csr, cp_ones), name="CuPy CSR", n_repeat=10)
cupy3 = benchmark(matmul, (a_cp_coo, cp_ones), name="CuPy COO", n_repeat=10)

print(cupy1)
print(cupy2)
print(cupy3)    

Then we benchmark UST.

In [ ]:
with Matmul(u_dia, u_ones, u_out) as mm:
    mm.plan()
    ust1 = benchmark(mm.execute, name="UST DIA", n_repeat=10)
    
with Matmul(u_csr, u_ones, u_out) as mm:
    mm.plan()
    ust2 = benchmark(mm.execute, name="UST CSR", n_repeat=10)    
    
with Matmul(u_coo, u_ones, u_out) as mm:
    mm.plan()
    ust3 = benchmark(mm.execute, name="UST COO", n_repeat=10)

print(ust1)
print(ust2)
print(ust3)   

And finally we plot the results.

Note how UST DIA format performs best, since it JIT compiles into a tailored kernel for that format. The UST COO outperforms CuPy COO by far (something the CuPy team could look into). The UST CSR slightly outperforms CuPy CSR, even though both use cuSPARSE as backend, simply because nvmath-python provides the plan-execute setup. You can increase/decrease the value of n above (and run back until here) to play with different sizes and see how that impacts the CPU overhead vs. actual GPU runtime characteristics. Overall, nvmath-python has less CPU overhead getting into the library than CuPy (visible for smaller `n`) while the kernels run faster to GPU as well (visible for larger `n`).

In [ ]:
import matplotlib.pyplot as plt

labels = [cupy1.name, cupy2.name, cupy3.name, ust1.name, ust2.name, ust3.name]
runtimes = [  # seconds to useconds
    min(cupy1.gpu_times[0]) * 1e6,
    min(cupy2.gpu_times[0]) * 1e6,
    min(cupy3.gpu_times[0]) * 1e6,
    min(ust1.gpu_times[0]) * 1e6,
    min(ust2.gpu_times[0]) * 1e6,
    min(ust3.gpu_times[0]) * 1e6,
]
bar_colors = ["#306998", "#306998", "#306998", "#76B900", "#76B900", "#76B900"]
hatch = ["+", ".", "/", "+", ".", "/"]

plt.figure(figsize=(8, 6))
plt.title(f"SpMV runtime for n={n}")
plt.bar(labels, runtimes, color=bar_colors, hatch=hatch)
plt.ylabel("Execution Time (us) - ** LOG SCALE **")
plt.yscale("log")
plt.xticks(rotation=45)

plt.show()

## Amortization of Planning Cost (even for expensive one-time JIT/LTO)

So far, we have compared the regular execution time of CuPy with the execution time of nvmath-python *after* planning. This is a fair comparison for cases where a single planning phase is followed by many subsequent executions. However, for a generally fair comparison, below we plot the CuPy DIA performance against the UST DIA performance for a completely cold cache planning phase. A full JIT/LTO compilation of a novel kernel can easily take 25000us (it is hard to measure exactly in this notebook because this is done only once, even between new planning phases; also, typically the planning phase is much cheaper, since it often can use the cuSPARSE library directly). To show a typical worst-case when the amortization must be taken into account, below we plot the measured CuPy DIA against the measure UST DIA performance using a hard-coded overhead (you can change this to see the differences). The plot now shows how many iterations are required to fully benefit from the UST when planning time is at a premium as well.


In [ ]:
import numpy as np

cupy_dia = min(cupy1.gpu_times[0]) * 1e6
ust_dia = min(ust1.gpu_times[0]) * 1e6
planning_time = 25000     # put JIT/LTO estimate here (in us)

print(cupy_dia, 'vs', planning_time, ust_dia)

i_values = np.arange(1, 100)
values1 = np.full_like(i_values, cupy_dia, dtype=float)
values2 = ust_dia + planning_time / i_values

plt.figure(figsize=(10, 6))
plt.plot(i_values, values1, 'b-', linewidth=2, label="CuPy DIA")
plt.plot(i_values, values2, 'r-', marker='o', linewidth=2, markersize=6, label="UST DIA (amortized)")
plt.xlabel('iterations', fontsize=12)
plt.ylabel('runtime (us)', fontsize=12)
plt.ylim(0, 2*cupy_dia)  # pick a region that puts the constant cupy_dia in the middle
plt.title("Amortized planning vs execution time (when including JIT/LTO time)", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## Novel Formats

Besides supporting common sparse storage formats, the DSL of the UST allows for defining your own nifty new format, and all prior operations (like matmul) work out of the box using the JITted codegen path when no library support exists. Various less common sparse storage formats are already pre-defined the NamedFormats class, which is a good starting point to get more familiar with this DSL. Printing each format (which is really a Python data structure) shows a human readable representation of the DSL for that format.

In [ ]:
from nvmath.sparse.ust import NamedFormats

print(NamedFormats.COO)
print(NamedFormats.CSR)
print(NamedFormats.CSC)
print(NamedFormats.DCSR)
print(NamedFormats.DCSC)
print(NamedFormats.BSRRight((8,8)))
print(NamedFormats.BSCRight((8,8)))

## A Brief Introduction to the UST DSL

Each format consists of a list of **dimensions** that are mapped onto a list of **levels**. Here the dimensions consists of a row index `i` and column index `j`, but this would a singleton sequence like `[i]` or `[first]` for vectors and a sequence such as `[i, j, k]` for tensors. The names for the dimensions are arbitrary, and can be chosen by the user based on the the application domain. The levels denote what is stored (like `i` or `i // 8`), as well as level formats (like `dense` or `compressed`) and level properties (like `nonunique`).

For example, let's look at the CSR format.

`[i, j] -> (i: <LevelFormat.DENSE>, j: <LevelFormat.COMPRESSED>)`

This format maps two dimensions `[i, j]` to two levels in the same order, using dense storage for `i`, i.e. each row is represented, but compressed storage for `j`. i.e. within each row, we only store values and column indices. The CSC format essentially does the same thing, but permutes the row and column index within the levels, forcing a compressed storage along the columns rather than along the rows.

Now let's now take a look at how to define a BSR format with 8x8 blocks (called BSRRight(8,8)).

`[i, j] -> ((i // 8): <LevelFormat.DENSE>, (j // 8): <LevelFormat.COMPRESSED>, (i % 8): <LevelFormat.DENSE>, (j % 8): <LevelFormat.DENSE>)`

A breakdown of this DSL expression is as follows:

- As before, we have two dimensions since we are storing matrices: `[i, j]`
- These dimensions denote the logical matrix indices: row index and column index
- But now, we have four levels, representing physical memory indices
- The first two levels `i // 8` and `j // 8` essentialy yield a block index, stored `(dense, compressed)` to get a CSR-flavored storage of nonzero blocks
- The next two levels `i % 8` and `j % 8` map these blocks to an 8x8 index space within each block, stored `(dense, dense)`, all elements, even the zeros, are stored without any futher metadata

Note that if the last two levels were also stored `(dense, compressed)`, the format would define an interesting hiarchical CSR structure, which is sometimes used to reduce the number of bits required to store indices.

To learn more about the DSL, have a look at the [nvmath-python UST DSL documentation](https://docs.nvidia.com/cuda/nvmath-python/0.9.0/host-apis/sparse/index.html#ust-dsl) as well as the implementation in the [tensor_format.py ](https://github.com/NVIDIA/nvmath-python/blob/main/nvmath/sparse/ust/tensor_format.py) file.

An existing UST can be converted into any of these formats with the following conversion call (note that this is no longer a zero-cost operation, since data needs to be reshuffled; also, the current conversion implementation is a prototype, and can be rather costly).

In [ ]:
# For large n, this may run for a while (see "kernel busy" in bottom bar).
# Just wait for the output to appear (or reduce n above and run until here).
u_block = u_coo.convert(tensor_format=NamedFormats.BSRRight((8, 8)))
print(u_block)

## Exercise

Convert the UST for u_coo to DCSR and perform the matmul benchmark on this new format. Compare the performance with the UST implementation of the other three formats shown earlier (and keep in mind that this uses a new JIT/LTO kernel, since no backend library supports the doubly-compressed format).

In [ ]:
# Convert u_coo to u_dcsr.

# Create a Matmul object, plan, and benchmark.

# Plot DIA/CSR/COO/DCSR results.

## Bonus Exercise

Let's design a very novel sparse storage scheme! First consider a very special sparse matrix, where nonzeros are clustered in regular strips of 4.

In [ ]:
strips = cp.array([ [1,  2,  3,  4,  0,  0,  0,  0,  0,  0,  0,  0], 
                    [0,  0,  0,  0,  0,  0,  0,  0,  5,  6,  7,  8], 
                    [0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
                    [0,  0,  0,  0,  9, 10, 11, 12,  0,  0,  0,  0]], dtype=cp.float32)

Now let's convert this dense array into a UST first (yes, the DSL can even describe dense formats!), and then into CSR, DCSR, and COO. Printing each reports how the data is stored and how many bytes are required.

In [ ]:
u_dense = Tensor.from_package(strips)
u_csr = u_dense.convert(tensor_format=NamedFormats.CSR)
u_dcsr = u_dense.convert(tensor_format=NamedFormats.DCSR)
u_coo = u_dense.convert(tensor_format=NamedFormats.COO)
print(u_dense)
print(u_csr)
print(u_dcsr)
print(u_coo)

Now let's explore if we can do better by designing our own novel data structure that stores 1x4 strips only 

**Hint:** get inspiration from the BSR format we explored above, but use three levels instead of four levels

In [ ]:
# Design a strip DSL for the UST.

# Convert the array above into that format.

# Print the result to see how many bytes are used.


## Conclusion

Even though we could not cover all details in this notebook, hopefully the material presented here has piqued your interest into learning more about the Universal Sparse Tensor!